# Few-Shot Prompting

Here, we'll continue to focus on out week two example: **prompt engineering**. Since we have our few-shot examples and setlist of skits ready, we'll start by testing few-shot prompting exclusively in this notebook. This means:

* Giving our chosen LLM system instructions *with* ready-made examples
* Providing a test set of user inputs, having it output a humor score, then grading it ourselves
* And recording takeaways that can be used to compare zero-shot prompting

For context, we'll be using Gemini 3 Flash for this step of the project.

In [ ]:
# Install the official Google Generative AI SDK ('-q' silences the output, '-U' ensures the most up-to-date version)
!pip install -q -U google-generativeai

In [35]:
from google.colab import userdata
from google import genai
import re # regex: extracting specific number
import io
import csv
from google.genai import types

#Fetch API key
GEMINI_API_KEY = userdata.get('Gemini')
client = genai.Client(api_key=GEMINI_API_KEY)

In [23]:
#upload comedy data
from google.colab import files
uploaded = files.upload()

Saving DS 340 Final Project Data.csv to DS 340 Final Project Data.csv


In [36]:
#csv read, extract joke and rating

def fewshot_data_split(uploaded):

  # grab and parse csv
  file_name = list(uploaded.keys())[0]
  file_content = uploaded[file_name]
  csv_text = file_content.decode('latin-1')
  csv_reader = csv.reader(io.StringIO(csv_text, newline=''))
  next(csv_reader)

  training_sets = []
  testing_sets = []

  # split up the training set and testing set data
  for row in csv_reader:
    if not row:
      continue
    try:
      row_id = int(row[0])
      joke = row[1]
      rating = str(row[2])

      #training sets
      rating_2_train = (11 <= row_id <= 34)
      rating_1_train = (46 <= row_id <= 67)
      rating_0_train = (79 <= row_id <= 100)

      #testing sets
      rating_2_test = (1 <= row_id <= 10)
      rating_1_test = (35 <= row_id <= 45)
      rating_0_test = (68 <= row_id <= 78)

      if rating_2_train or rating_1_train or rating_0_train:
        training_sets.append(types.Content(role='user', parts=[types.Part.from_text(text=f"Text: {joke}")]))
        training_sets.append(types.Content(role='model', parts=[types.Part.from_text(text=rating)]))

      elif rating_2_test or rating_1_test or rating_0_test:
        testing_sets.append({
            "id": row_id,
            "joke": joke,
            "rating": rating
            })

    except (ValueError, IndexError):
      continue


  return training_sets, testing_sets


In [37]:
#Initialize model

def few_shot_prediction(training_sets, testing_sets):
  y_true = []
  y_pred = []
  instruction = ("You are a comedy writer and your job is to evaluate how funny the given piece of text is."
  "Classify the input into the following categories: 0(not funny), 1(neutral), 2(funny)."
  "The classification can only be numerical and nothing else.")

  for test in testing_sets:
    y_true.append(int(test['rating'])) #append true labels
    test_input = types.Content(role='user', parts=[types.Part.from_text(text=f"Text: {test['joke']}")])

    #combine all training examples + one test input
    full_context = training_sets + [test_input]

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=full_context,
        config=types.GenerateContentConfig(
        temperature = 1,
        system_instruction = instruction,
        ),
      )
    prediction = response.text.strip()
    y_pred.append(int(prediction))

  return y_true, y_pred


In [38]:
# Passes the data through the LLM using few-shot prompt & returns lists of actual vs. predicted labels
def few_shot_prediction(training_sets, testing_sets):
    y_true = []
    y_pred = []
    instruction = (
        "You are a comedy writer. Evaluate the humor of the text. "
        "Classify as: 0 (not funny), 1 (neutral), or 2 (funny). "
        "Output ONLY the numerical digit (0, 1, or 2)."
    )

    for test in testing_sets:
        y_true.append(int(test['rating']))

        # Create the test input
        test_input = types.Content(
            role='user',
            parts=[types.Part.from_text(text=f"Text: {test['joke']}")]
        )

        # Combine examples + test input so the model has context
        full_context = training_sets + [test_input]

        try:
            response = client.models.generate_content(
                model="gemini-3-flash-preview",
                contents=full_context,
                config=types.GenerateContentConfig(
                    system_instruction=instruction,
                    temperature=0,
                ),
            )

            # Parsing
            prediction_text = response.text.strip()

            # Find the first digit (0, 1, or 2) in the response
            match = re.search(r'[0-2]', prediction_text)

            if match:
                y_pred.append(int(match.group()))
            else:
                # Debug: print out if the model returns something off
                print(f"Warning: Model returned unexpected text for ID {test['id']}: {prediction_text}")
                y_pred.append(0)

        except Exception as e:
            print(f"Error predicting ID {test['id']}: {e}")
            y_pred.append(0)

    return y_true, y_pred

In [39]:
# accuracy evaluation report
from sklearn.metrics import classification_report

# extract data from the pre-existed dictionary
training_data, testing_data = fewshot_data_split(uploaded)

# run prediction
y_true, y_pred = few_shot_prediction(training_data, testing_data)

# print report
print("\n" + "=" * 50)
print("Few-shot Classification Report")
print("=" * 50)
print(classification_report(
    y_true,
    y_pred,
    labels = [0, 1, 2],
    target_names = ["0 (not funny)", "1 (neutral)", "2 (funny)"],
    zero_division = 0
))



Few-shot Classification Report
               precision    recall  f1-score   support

0 (not funny)       0.38      0.82      0.51        11
  1 (neutral)       0.00      0.00      0.00        11
    2 (funny)       0.25      0.20      0.22        10

     accuracy                           0.34        32
    macro avg       0.21      0.34      0.25        32
 weighted avg       0.21      0.34      0.25        32

